<a href="https://www.kaggle.com/code/dheriisousa/02-pytorch-validation-ipynb?scriptVersionId=323996340" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.4 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from medmnist import PathMNIST

# 1. FIXANDO AS SEEDS (Regra de Ouro da Reprodutibilidade)
# O professor tira 10% da nota se não houver seed fixada!
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Carregando dataset 28x28 para o PyTorch...")
train_dataset = PathMNIST(split="train", size=28, download=True)
val_dataset = PathMNIST(split="val", size=28, download=True)

# 2. Preparando os tensores
# PyTorch usa tensores (torch.tensor) em vez de arrays (np.array)
X_train_np = train_dataset.imgs.reshape(train_dataset.imgs.shape[0], -1) / 255.0
X_val_np = val_dataset.imgs.reshape(val_dataset.imgs.shape[0], -1) / 255.0

# Convertendo para Tensores PyTorch (Float32 para imagens, Long para rótulos)
X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(train_dataset.labels.squeeze(), dtype=torch.long)

X_val = torch.tensor(X_val_np, dtype=torch.float32)
y_val = torch.tensor(val_dataset.labels.squeeze(), dtype=torch.long)

# 3. Definindo a MLP idêntica usando PyTorch (Orientação a Objetos)
class PyTorchMLP(nn.Module):
    def __init__(self):
        super(PyTorchMLP, self).__init__()
        self.camadas = nn.Sequential(
            nn.Linear(2352, 128), # Primeira camada oculta
            nn.ReLU(),            # Ativação ReLU
            nn.Linear(128, 9)     # Camada de Saída (Sem Softmax explícito!)
        )
        
        # Copiando a mesma inicialização He (Kaiming) que usamos no NumPy
        nn.init.kaiming_normal_(self.camadas[0].weight, nonlinearity='relu')
        nn.init.zeros_(self.camadas[0].bias)
        nn.init.xavier_normal_(self.camadas[2].weight)
        nn.init.zeros_(self.camadas[2].bias)

    def forward(self, x):
        return self.camadas(x)

modelo_pt = PyTorchMLP()

# 4. Configurando Função de Perda e Otimizador
# Exatamente os mesmos parâmetros da Etapa 1
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(modelo_pt.parameters(), lr=0.05, momentum=0.9)

# 5. Loop de Treinamento
epocas = 15
batch_size = 256
n_amostras = X_train.shape[0]

print("\nIniciando treinamento no PyTorch...")

for epoca in range(epocas):
    # Coloca o modelo em modo de treinamento
    modelo_pt.train()
    
    # Embaralhando os índices manualmente para imitar nossa lógica NumPy
    indices = torch.randperm(n_amostras)
    X_train_shuffled = X_train[indices]
    y_train_shuffled = y_train[indices]
    
    loss_epoca = 0.0
    
    for i in range(0, n_amostras, batch_size):
        X_batch = X_train_shuffled[i:i + batch_size]
        y_batch = y_train_shuffled[i:i + batch_size]
        
        # Zera os gradientes acumulados no otimizador
        optimizer.zero_grad()
        
        # Forward Pass
        predicoes = modelo_pt(X_batch)
        
        # Calcula o erro (Loss)
        loss = criterion(predicoes, y_batch)
        
        # Backward Pass (O PyTorch faz toda aquela matemática pra gente aqui)
        loss.backward()
        
        # Atualiza os pesos com SGD + Momentum
        optimizer.step()
        
        loss_epoca += loss.item()
        
    # Avaliação
    modelo_pt.eval() # Coloca em modo de avaliação (desliga dropout/batchnorm se houvesse)
    with torch.no_grad(): # Desliga o rastreio de gradientes para economizar memória
        val_preds = modelo_pt(X_val)
        acertos = (torch.argmax(val_preds, dim=1) == y_val).sum().item()
        acc_val = (acertos / len(y_val)) * 100
        
    loss_media = loss_epoca / (n_amostras / batch_size)
    print(f"Época {epoca+1}/{epocas} - Loss: {loss_media:.4f} - Acurácia Validação: {acc_val:.2f}%")

Carregando dataset 28x28 para o PyTorch...


100%|██████████| 206M/206M [00:10<00:00, 19.3MB/s]



Iniciando treinamento no PyTorch...
Época 1/15 - Loss: 2.1962 - Acurácia Validação: 13.54%
Época 2/15 - Loss: 2.1880 - Acurácia Validação: 14.31%
Época 3/15 - Loss: 2.1879 - Acurácia Validação: 14.31%
Época 4/15 - Loss: 2.1881 - Acurácia Validação: 14.31%
Época 5/15 - Loss: 2.1881 - Acurácia Validação: 14.31%
Época 6/15 - Loss: 2.1880 - Acurácia Validação: 14.31%
Época 7/15 - Loss: 2.1879 - Acurácia Validação: 14.31%
Época 8/15 - Loss: 2.1881 - Acurácia Validação: 13.53%
Época 9/15 - Loss: 2.1880 - Acurácia Validação: 14.31%
Época 10/15 - Loss: 2.1878 - Acurácia Validação: 14.31%
Época 11/15 - Loss: 2.1877 - Acurácia Validação: 14.31%
Época 12/15 - Loss: 2.1876 - Acurácia Validação: 14.31%
Época 13/15 - Loss: 2.1880 - Acurácia Validação: 14.31%
Época 14/15 - Loss: 2.1880 - Acurácia Validação: 14.31%
Época 15/15 - Loss: 2.1881 - Acurácia Validação: 14.32%
